In [112]:
import torch
import numpy as np


%load_ext autoreload
%autoreload 2

# Global libraries
import os
import torch
import pandas as pd
from torch.utils.data import DataLoader
from collections import defaultdict
from PIL import ImageFont
from torchinfo import summary

# Local libraries enivironment loading
import sys
from pathlib import Path
path_Project = Path(os.getcwd()).parent.parent
path_Git = path_Project.parent

def IncludeDirectory(path, index = 0, indexStop = -1):
    if(os.path.exists(path) and (index <= indexStop or indexStop == -1)):
        print(path)
        sys.path.append(path)
        children = os.listdir(path)
        for i in range(len(children)):
            pathChild = os.path.join(path,children[i])
            if(os.path.isdir(pathChild)):
                IncludeDirectory(pathChild, index + 1, indexStop)

# Activate the function
IncludeDirectory(path_Git,0,1)

from src.config_presets.tools.get_config import get_config, load_config
from src.utils.set_random_seed import set_random_seed



C:\Users\S.P.M. de Vette\AppData\Local\Temp\ipykernel_10744\2853916192.py:11: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was too old on your system - pyarrow 10.0.1 is the current minimum supported version as of this release.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


c:\Users\S.P.M. de Vette\OneDrive - UMCG\Desktop
c:\Users\S.P.M. de Vette\OneDrive - UMCG\Desktop\pred_RT
c:\Users\S.P.M. de Vette\OneDrive - UMCG\Desktop\pred_RT_results


In [147]:
true_labels = torch.tensor([[1, 0, 1, 1, 0],
                             [0, 1, 1, 0, 1],
                             [1, 0, 0, 1, 0],
                             [1, 0, 0, 1, 0],
                             [0, 1, 0, 0, 1],
                             [0, 1, 1, 0, 1],
                             [1, 0, 0, 1, 0],
                             [1, 0, 0, 1, 0],
                             [0, 1, 0, 0, 1],

                             ], dtype=torch.float32)
# true_labels_2 = torch.tensor([[0, 1, 1, 0, 0],
#                              [1, 0, 1, 1, 0],
#                              [0, 1, 0, 0, 0],
#                              [0, 1, 0, 0, 0]])

model_preds = torch.tensor([[0.4, 0.6, 0.3, 0.8, 0.1],
                            [0.2, 0.5, 0.7, 0.3, 0.4],
                            [0.6, 0.9, 0.2, 0.5, 0.3],
                            [0.7, 0.2, 0.1, 0.6, 0.4],
                            [0.2, 0.5, 0.7, 0.3, 0.4],
                            [0.6, 0.9, 0.2, 0.5, 0.3],
                            [0.2, 0.5, 0.7, 0.3, 0.4],
                            [0.6, 0.9, 0.2, 0.5, 0.3],
                            [0.4, 0.6, 0.3, 0.8, 0.1]], dtype=torch.float32)
model_preds_as_logits = torch.log(model_preds / (1 - model_preds))


In [148]:
true_labels.shape

torch.Size([9, 5])

In [149]:
model_preds.shape

torch.Size([9, 5])

In [150]:
def mixup_labels(true_labels, alpha):
    batch_size = true_labels.size()[0]
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    
    indicies = torch.randperm(batch_size, device=true_labels.device)
    
    mixed_x = lam * true_labels + (1 - lam) * true_labels[indicies, :]
    #y_a, y_b = true_labels, true_labels[indicies]
    return mixed_x, lam, indicies

mixed_labels, lam, mix_indices = mixup_labels(true_labels, alpha=0.2)

mixed_labels

tensor([[1.0000, 0.0000, 0.8841, 1.0000, 0.0000],
        [0.1159, 0.8841, 0.8841, 0.1159, 0.8841],
        [1.0000, 0.0000, 0.1159, 1.0000, 0.0000],
        [0.8841, 0.1159, 0.0000, 0.8841, 0.1159],
        [0.0000, 1.0000, 0.0000, 0.0000, 1.0000],
        [0.0000, 1.0000, 1.0000, 0.0000, 1.0000],
        [1.0000, 0.0000, 0.0000, 1.0000, 0.0000],
        [0.8841, 0.1159, 0.1159, 0.8841, 0.1159],
        [0.1159, 0.8841, 0.0000, 0.1159, 0.8841]])

In [151]:
BCE_loss = torch.nn.BCEWithLogitsLoss(reduction='none')

In [152]:
easy_loss = BCE_loss(model_preds_as_logits, mixed_labels)
print(easy_loss)
print(easy_loss.mean())

tensor([[0.9163, 0.9163, 1.1058, 0.2231, 0.1054],
        [0.3838, 0.6931, 0.4549, 0.4549, 0.8693],
        [0.5108, 2.3026, 0.3838, 0.6931, 0.3567],
        [0.4549, 0.3838, 0.1054, 0.5578, 0.5578],
        [0.2231, 0.6931, 1.2040, 0.3567, 0.9163],
        [0.9163, 0.1054, 1.6094, 0.6931, 1.2040],
        [1.6094, 0.6931, 1.2040, 1.2040, 0.5108],
        [0.5578, 2.0479, 0.3838, 0.6931, 0.4549],
        [0.5578, 0.5578, 0.3567, 1.4488, 2.0479]])
tensor(0.7706)


In [153]:
def mixup_criterion(criterion, pred, y_a, y_b, lam):
    part1 = criterion(pred, y_a)
    part2 = criterion(pred, y_b)

    return lam * part1 + (1 - lam) * part2
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

hard_loss = mixup_criterion(BCE_loss, model_preds_as_logits, true_labels, true_labels[mix_indices], lam)

In [154]:
print(hard_loss)
print(hard_loss.mean())

tensor([[0.9163, 0.9163, 1.1058, 0.2231, 0.1054],
        [0.3838, 0.6931, 0.4549, 0.4549, 0.8693],
        [0.5108, 2.3026, 0.3838, 0.6931, 0.3567],
        [0.4549, 0.3838, 0.1054, 0.5578, 0.5578],
        [0.2231, 0.6931, 1.2040, 0.3567, 0.9163],
        [0.9163, 0.1054, 1.6094, 0.6931, 1.2040],
        [1.6094, 0.6931, 1.2040, 1.2040, 0.5108],
        [0.5578, 2.0479, 0.3838, 0.6931, 0.4549],
        [0.5578, 0.5578, 0.3567, 1.4488, 2.0479]])
tensor(0.7706)


In [155]:
lam

0.8841012018466196

In [156]:
from src.evaluation.calculate_auc import calculate_auc_multi

config = {'columns': {'label': ["Aspiration_M06", "Dysphagia_M06", "Sticky_M06", "Taste_M06", "Xerostomia_M06" ]}}

true_labels_dict = {endpoint : true_labels[:, j] for (j, endpoint) in enumerate(config['columns']['label'])}
model_preds_dict = {endpoint : model_preds[:, j] for (j, endpoint) in enumerate(config['columns']['label'])}

AUC = calculate_auc_multi(model_preds_dict, true_labels_dict, config)

AUC

{'Aspiration_M06': 0.725,
 'Dysphagia_M06': 0.475,
 'Sticky_M06': 0.583,
 'Taste_M06': 0.625,
 'Xerostomia_M06': 0.525}

In [160]:
mixed_labels_dict = {endpoint : mixed_labels[:, j] for (j, endpoint) in enumerate(config['columns']['label'])}
model_preds_dict = {endpoint : model_preds[:, j] for (j, endpoint) in enumerate(config['columns']['label'])}

AUC = calculate_auc_multi(model_preds_dict, mixed_labels_dict, config)

AUC

{'Aspiration_M06': -1,
 'Dysphagia_M06': -1,
 'Sticky_M06': -1,
 'Taste_M06': -1,
 'Xerostomia_M06': -1}

In [159]:
mixed_labels

tensor([[1.0000, 0.0000, 0.8841, 1.0000, 0.0000],
        [0.1159, 0.8841, 0.8841, 0.1159, 0.8841],
        [1.0000, 0.0000, 0.1159, 1.0000, 0.0000],
        [0.8841, 0.1159, 0.0000, 0.8841, 0.1159],
        [0.0000, 1.0000, 0.0000, 0.0000, 1.0000],
        [0.0000, 1.0000, 1.0000, 0.0000, 1.0000],
        [1.0000, 0.0000, 0.0000, 1.0000, 0.0000],
        [0.8841, 0.1159, 0.1159, 0.8841, 0.1159],
        [0.1159, 0.8841, 0.0000, 0.1159, 0.8841]])

In [119]:
true_labels[0]

tensor([1., 0., 1., 1., 0.])

In [118]:
model_preds_dict

{'Xero_M06': tensor([[0.4000, 0.6000, 0.3000, 0.8000, 0.1000],
         [0.2000, 0.9000, 0.7000, 0.3000, 0.4000],
         [0.6000, 0.1000, 0.2000, 0.5000, 0.3000],
         [0.7000, 0.2000, 0.1000, 0.6000, 0.4000]])}

In [116]:
true_labels

tensor([[1., 0., 1., 1., 0.],
        [0., 1., 1., 0., 0.],
        [1., 0., 0., 1., 0.],
        [1., 0., 0., 1., 0.]])